In [31]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit, minimize
import astropy.units as u
from astropy.io import fits
from astropy.table import Table, vstack, join
from scipy.stats import ttest_rel
from IPython.display import display, Math
from pathlib import Path

with fits.open("/nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits") as hdul:
        data = hdul[1].data
TABLE = Table(data)

In [32]:
def open_catalogue_data(path):
  """
  Reads in a catalogue of data from a FITS file and returns the data as an Astropy Table.
  """

  hdul = fits.open(path)
  hdu_names = [hdu.name for hdu in hdul]

  # sex_cat = [hdu for hdu, name in zip(hdul, hdu_names) if name == "OBJECTS"][0]
    
  # EPOCHS series (Conselice+24, Adams+24, Austin+25, Harvey+25)

  sex_tab = Table.read(path, hdu = "OBJECTS")
  
  eazy_tab = Table.read(path, hdu = "EAZY_SFHZ_BLUE_AGN")
  eazy_tab_colnames = eazy_tab.colnames

  eazy_properties_tab = Table.read(path, hdu = 'PROPERTIES_EAZY_SFHZ_BLUE_AGN')
  eazy_properties_tab_colnames = eazy_properties_tab.colnames
  # beta/Muv

  selection_tab = Table.read(path, hdu = "SELECTION")
  selection_tab_colnames = selection_tab.colnames

  # EPOCHS_good_EAZY_sfhz_blue_agn_0.32as == True

  bagpipes_tab = Table.read(path, hdu = 'BAGPIPES_SFH_CONT_BURSTY_ZEAZYSFHZBLUEAGN_3.0,10.0MYR_CALZETTI_LOG_10_Z_LOG_10_BPASS_ZGAUSS_3.0SIG')
  bagpipes_tab_colnames = bagpipes_tab.colnames

  # only run on EPOCHS_good_EAZY_sfhz_blue_agn_0.32as == True AND z > 6.5
  # galaxy properties
  
  return sex_tab, eazy_tab, eazy_properties_tab, selection_tab, bagpipes_tab

In [33]:
def mass_lim_eq(stellar_mass, m_AB, depth):
    """
    stellar mass from BAGPIPES
    m_AB is brightness in F444W
    depth is m_AB limit
    Equation originally from Pozetti 2010.
    """
    log_M_lim = np.log10(stellar_mass) + 0.4 * (m_AB - depth)

    mass_lim = 10**log_M_lim

    return mass_lim

In [ ]:
def find_depth(survey):

    if survey in ["CEERSP1", "CEERSP2", "CEERSP3", "CEERSP4", "CEERSP5", "CEERSP6", "CEERSP7", "CEERSP8", "CEERSP9", "CEERSP10"]:

            path = f"/raid/scratch/work/austind/GALFIND_WORK/Depths/Depth_tables/v14/{survey}/{survey}_depths.ecsv"

    elif survey in ["PRIMER-COSMOS","PRIMER-UDS"]:
            
            path = f"/raid/scratch/work/austind/GALFIND_WORK/Depths/Depth_tables/v12/{survey}/{survey}_depths.ecsv"


    elif survey in ["JADES-DR3-GS-East", "JADES-DR3-GS-North", "JADES-DR3-GS-South", "JADES-DR3-GS-West", "JADES-DR3-GN-Parallel", "JADES-DR3-GN-Medium", "JADES-DR3-GN-Deep"]:

            path = f"/raid/scratch/work/austind/GALFIND_WORK/Depths/Depth_tables/v13/{survey}/{survey}_depths.ecsv"
    else:
            print(f"Galaxy {id} could not be found in these surveys")
            print(f"Check survey {survey}")

    path = Path(path)
    if path.exists():
        depth_table = Table.read(path, format="ecsv")
        mask = (depth_table["filter"] == "F444W") & (depth_table["mode"] == 1)
        depth = depth_table[mask]['mean_depth']
        print(depth)
        return depth 
    else:
        print(f"{path} could not be found.")

       

In [35]:
find_depth("JADES-DR3-GN-Medium")

/raid/scratch/work/austind/GALFIND_WORK/Depths/Depth_tables/v13/JADES-DR3-GN-Medium/JADES-DR3-GN-Medium_depths.escv could not be found.
